In [8]:
import os

print(os.listdir(r"D:\Project\db"))

['hr_interview_questions_dataset.json']


In [9]:
import pandas as pd

df = pd.read_json(
    r"D:\Project\db\hr_interview_questions_dataset.json"
)

print(df.shape)
print(df.columns.tolist())
print(df.head(3).to_string())



(2500000, 8)
['question', 'category', 'role', 'experience', 'difficulty', 'source_type', 'ideal_answer', 'keywords']
                                                                  question             category             role experience difficulty source_type                                                                                                                                                                                                               ideal_answer             keywords
0  Tell me about a time you had to learn something completely new quickly.         Adaptability  DevOps Engineer    fresher       Easy  Open-Ended  I'm always eager to learn and embrace change as a way to improve. For example, I once had to switch to a new tech stack and picked it up quickly. Tell me about a time you had to learn something completely new quickly.   [flexible, change]
1        Describe a time you handled a difficult situation professionally.  Conflict Resolution  Product Mana

In [10]:
print(df.dtypes)


question           str
category           str
role               str
experience         str
difficulty         str
source_type        str
ideal_answer       str
keywords        object
dtype: object


In [11]:
print(df.dtypes)
print(df.head(2))


question           str
category           str
role               str
experience         str
difficulty         str
source_type        str
ideal_answer       str
keywords        object
dtype: object
                                            question             category  \
0  Tell me about a time you had to learn somethin...         Adaptability   
1  Describe a time you handled a difficult situat...  Conflict Resolution   

              role experience difficulty source_type  \
0  DevOps Engineer    fresher       Easy  Open-Ended   
1  Product Manager    2 years       Hard  Behavioral   

                                        ideal_answer             keywords  
0  I'm always eager to learn and embrace change a...   [flexible, change]  
1  When faced with conflict, I approach it calmly...  [problem, disagree]  


In [6]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="127.0.0.1",
        port=5433,
        database="ragdb",
        user="postgres",
        password="postgres"
    )
    print("Connected successfully!")

except psycopg2.Error as e:
    print("POSTGRESQL ERROR:")
    print(str(e))


Connected successfully!


In [7]:
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS interview_questions (
    id BIGSERIAL PRIMARY KEY,
    question TEXT,
    category TEXT,
    role TEXT,
    experience TEXT,
    difficulty TEXT,
    source_type TEXT,
    ideal_answer TEXT,
    keywords TEXT
);
""")

conn.commit()

print("Table created successfully!")

Table created successfully!


In [ ]:
from psycopg2.extras import execute_values

batch_size = 10_000

insert_query = """
INSERT INTO interview_questions
(question, category, role, experience, difficulty, source_type, ideal_answer, keywords)
VALUES %s
"""

for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start + batch_size]

    records = [
        tuple(row)
        for row in batch[
            [
                "question",
                "category",
                "role",
                "experience",
                "difficulty",
                "source_type",
                "ideal_answer",
                "keywords"
            ]
        ].itertuples(index=False, name=None)
    ]

    execute_values(cur, insert_query, records)
    conn.commit()

    print(f"Imported {min(start + batch_size, len(df)):,} / {len(df):,}")

print("Dataset import completed!")

Imported 10,000 / 2,500,000
Imported 20,000 / 2,500,000
Imported 30,000 / 2,500,000
Imported 40,000 / 2,500,000
Imported 50,000 / 2,500,000
Imported 60,000 / 2,500,000
Imported 70,000 / 2,500,000
Imported 80,000 / 2,500,000
Imported 90,000 / 2,500,000
Imported 100,000 / 2,500,000
Imported 110,000 / 2,500,000
Imported 120,000 / 2,500,000
Imported 130,000 / 2,500,000
Imported 140,000 / 2,500,000
Imported 150,000 / 2,500,000
Imported 160,000 / 2,500,000
Imported 170,000 / 2,500,000
Imported 180,000 / 2,500,000
Imported 190,000 / 2,500,000
Imported 200,000 / 2,500,000
Imported 210,000 / 2,500,000
Imported 220,000 / 2,500,000
Imported 230,000 / 2,500,000
Imported 240,000 / 2,500,000
Imported 250,000 / 2,500,000
Imported 260,000 / 2,500,000
Imported 270,000 / 2,500,000
Imported 280,000 / 2,500,000
Imported 290,000 / 2,500,000
Imported 300,000 / 2,500,000
Imported 310,000 / 2,500,000
Imported 320,000 / 2,500,000
Imported 330,000 / 2,500,000
Imported 340,000 / 2,500,000
Imported 350,000 / 2,50

KeyboardInterrupt: 

In [4]:
print(records)

NameError: name 'records' is not defined

In [ ]:
cur.execute("SELECT COUNT(*) FROM interview_questions;")

count = cur.fetchone()[0]

print(f"Rows in PostgreSQL: {count:,}")

Rows in PostgreSQL: 2,500,000


In [ ]:
cur.execute("""
SELECT extname, extversion
FROM pg_extension
WHERE extname = 'vector';
""")

print(cur.fetchall())

[('vector', '0.8.6')]


In [ ]:
import sys
print(sys.version)


3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]


In [ ]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\Nitro\AppData\Local\Programs\Python\Python312\python.exe
3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]


In [15]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

KeyboardInterrupt: 

In [20]:
conn.rollback()
print("Transaction rolled back")

Transaction rolled back


In [21]:
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'interview_questions';
""")

print(cur.fetchall())

[('id', 'bigint'), ('question', 'text'), ('category', 'text'), ('role', 'text'), ('experience', 'text'), ('difficulty', 'text'), ('source_type', 'text'), ('ideal_answer', 'text'), ('keywords', 'text')]


In [22]:
cur.execute("""
    ALTER TABLE interview_questions
    ADD COLUMN IF NOT EXISTS embedding vector(768);
""")

conn.commit()

print("Embedding column ready!")

Embedding column ready!


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

print("Nomic model loaded!")

d:\Project III\AI-ML\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<All keys matched successfully>


Nomic model loaded!


In [7]:
cur.execute("""
    SELECT id, question
    FROM interview_questions
    WHERE embedding IS NULL
    LIMIT 100;
""")

rows = cur.fetchall()

print("Rows found:", len(rows))

if rows:
    texts = [row[1] for row in rows]

    embeddings = model.encode(texts)

    for (row_id, question), embedding in zip(rows, embeddings):
        cur.execute(
            """
            UPDATE interview_questions
            SET embedding = %s::vector
            WHERE id = %s;
            """,
            (str(embedding.tolist()), row_id)
        )

    conn.commit()

    print(f"Successfully embedded and stored {len(rows)} rows.")
else:
    print("No rows found that need embeddings.")

Rows found: 100
Successfully embedded and stored 100 rows.


In [ ]:
import ollama

def rag_chat(user_question):

    print("\n" + "="*60)
    print("STEP 1: USER QUESTION")
    print("="*60)
    print(user_question)

    # 1. Create embedding
    print("\n" + "="*60)
    print("STEP 2: CREATING QUERY EMBEDDING WITH NOMIC")
    print("="*60)

    query_embedding = model.encode(user_question).tolist()

    print("Embedding created!")
    print("Dimensions:", len(query_embedding))
    print("First 10 values:", query_embedding[:10])

    # 2. Vector search
    print("\n" + "="*60)
    print("STEP 3: VECTOR SEARCH IN POSTGRESQL")
    print("="*60)

    cur.execute("""
        SELECT
            id,
            question,
            category,
            role,
            experience,
            difficulty,
            ideal_answer,
            keywords,
            1 - (embedding <=> %s::vector) AS similarity
        FROM interview_questions
        WHERE embedding IS NOT NULL
        ORDER BY embedding <=> %s::vector
        LIMIT 5;
    """, (
        str(query_embedding),
        str(query_embedding)
    ))

    results = cur.fetchall()

    print("Top 5 similar rows found:\n")

    for i, row in enumerate(results, 1):
        print(f"--- Result {i} ---")
        print("ID:", row[0])
        print("Question:", row[1])
        print("Similarity:", round(row[8], 4))

    # 3. Build context
    print("\n" + "="*60)
    print("STEP 4: BUILDING RAG CONTEXT")
    print("="*60)

    context = "\n\n".join([
        f"""
Question: {row[1]}
Category: {row[2]}
Role: {row[3]}
Experience: {row[4]}
Difficulty: {row[5]}
Ideal Answer: {row[6]}
Keywords: {row[7]}
"""
        for row in results
    ])

    print(context)

    # 4. Llama
    print("\n" + "="*60)
    print("STEP 5: SENDING CONTEXT TO LLAMA 3.2")
    print("="*60)

    prompt = f"""
You are an AI interview assistant.

Use the following interview knowledge to answer the user's question.

CONTEXT:
{context}

USER QUESTION:
{user_question}

Answer naturally and clearly.
If the context does not contain enough information, say so instead of making up information.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    print("\n" + "="*60)
    print("STEP 6: LLAMA RESPONSE")
    print("="*60)
    print(answer)

    return answer

while True:

    user_question = input("\nYou: ")

    if user_question.lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    rag_chat(user_question)


STEP 1: USER QUESTION
i ama hr manager ask one question

STEP 2: CREATING QUERY EMBEDDING WITH NOMIC
Embedding created!
Dimensions: 768
First 10 values: [-0.8617948293685913, 0.08907859027385712, -3.34738826751709, 1.4921647310256958, 0.7563446760177612, 0.5938078761100769, 0.44523027539253235, -0.31846073269844055, 0.10474680364131927, -0.4938502311706543]

STEP 3: VECTOR SEARCH IN POSTGRESQL
Top 5 similar rows found:

--- Result 1 ---
ID: 4
Question: Tell me about a conflict you had with a coworker and how you resolved it.
Similarity: 0.5465
--- Result 2 ---
ID: 71
Question: What would you do if you disagreed with your manager?
Similarity: 0.5387
--- Result 3 ---
ID: 52
Question: What would you do if you disagreed with your manager?
Similarity: 0.5387
--- Result 4 ---
ID: 40
Question: What would you do if you disagreed with your manager?
Similarity: 0.5387
--- Result 5 ---
ID: 46
Question: Why do you want to work at our company?
Similarity: 0.5107

STEP 4: BUILDING RAG CONTEXT

Ques

KeyboardInterrupt: Interrupted by user

In [10]:
import ollama


# ============================================================
# 1. VECTOR SEARCH
# ============================================================

def search_questions(query, limit=5):

    print("\n🔍 Searching PostgreSQL + pgvector...")

    # Create embedding for the query
    query_embedding = model.encode(query).tolist()

    print(f"   ✓ Nomic embedding created ({len(query_embedding)} dimensions)")

    # Vector similarity search
    cur.execute("""
        SELECT
            id,
            question,
            category,
            role,
            experience,
            difficulty,
            ideal_answer,
            keywords,
            1 - (embedding <=> %s::vector) AS similarity
        FROM interview_questions
        WHERE embedding IS NOT NULL
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (
        str(query_embedding),
        str(query_embedding),
        limit
    ))

    results = cur.fetchall()

    print(f"   ✓ Retrieved {len(results)} relevant questions")

    return results


# ============================================================
# 2. CREATE CONTEXT FROM DATABASE
# ============================================================

def build_context(results):

    context = "\n\n".join([
        f"""
Question: {row[1]}
Category: {row[2]}
Role: {row[3]}
Experience: {row[4]}
Difficulty: {row[5]}
Ideal Answer: {row[6]}
Keywords: {row[7]}
Similarity: {row[8]:.3f}
"""
        for row in results
    ])

    return context


# ============================================================
# 3. GENERATE FIRST INTERVIEW QUESTION
# ============================================================

def generate_first_question(role):

    results = search_questions(role)

    context = build_context(results)

    prompt = f"""
You are a professional AI interviewer.

The candidate is applying for:

ROLE:
{role}

Here is interview knowledge retrieved from the database:

{context}

Your task:

1. Act as the interviewer.
2. Ask ONE interview question.
3. Make it relevant to the candidate's role.
4. Start with a reasonable difficulty.
5. Do not give the answer.
6. Do not ask multiple questions.

Return ONLY the interview question.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


# ============================================================
# 4. EVALUATE CANDIDATE ANSWER
# ============================================================

def evaluate_answer(role, question, answer):

    print("\n🧠 Llama 3.2 is evaluating your answer...")

    # Search database using the question
    results = search_questions(question, limit=3)

    context = build_context(results)

    prompt = f"""
You are an expert technical interviewer.

Candidate role:
{role}

Interview question:
{question}

Candidate's answer:
{answer}

Relevant interview knowledge:
{context}

Evaluate the candidate's answer.

Return the evaluation in this exact structure:

SCORE: X/10

STRENGTHS:
- ...
- ...

WEAKNESSES:
- ...
- ...

FEEDBACK:
...

NEXT_DIFFICULTY:
easy / medium / hard

Do not invent information.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


# ============================================================
# 5. GENERATE NEXT QUESTION
# ============================================================

def generate_next_question(
    role,
    previous_question,
    previous_answer,
    evaluation,
    asked_questions
):

    print("\n🔍 Searching for the next relevant question...")

    # Search using the candidate's previous answer
    search_query = f"""
    Role: {role}

    Previous question:
    {previous_question}

    Candidate answer:
    {previous_answer}

    Evaluation:
    {evaluation}
    """

    results = search_questions(search_query, limit=5)

    context = build_context(results)

    already_asked = "\n".join(
        f"- {q}" for q in asked_questions
    )

    prompt = f"""
You are a professional adaptive AI interviewer.

Candidate role:
{role}

Previous question:
{previous_question}

Candidate's previous answer:
{previous_answer}

Evaluation:
{evaluation}

Relevant interview knowledge:
{context}

Questions already asked:
{already_asked}

Your task:

Ask the NEXT interview question.

Rules:

1. Ask exactly ONE question.
2. Do not repeat a previous question.
3. Make the question relevant to the candidate's role.
4. Use the previous answer to decide what to ask next.
5. If the candidate showed weakness, ask a useful follow-up.
6. If the candidate answered very well, increase the difficulty.
7. If appropriate, move to another relevant topic.
8. Do not give the answer.
9. Return ONLY the interview question.

"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


# ============================================================
# 6. START INTERVIEW
# ============================================================

def start_interview():

    print("\n")
    print("=" * 65)
    print("                  AI INTERVIEWER")
    print("=" * 65)

    print("\nAI: Welcome! I'm your AI interviewer.")

    # --------------------------------------------------------
    # Ask role
    # --------------------------------------------------------

    role = input(
        "\nAI: What position are you applying for?\nYou: "
    )

    if role.lower() in ["exit", "quit"]:
        print("\nInterview ended.")
        return

    print("\nAI: Great. I'll prepare the interview based on that role.")

    # --------------------------------------------------------
    # First question
    # --------------------------------------------------------

    question = generate_first_question(role)

    asked_questions = []

    asked_questions.append(question)

    print("\n")
    print("-" * 65)
    print("AI INTERVIEWER")
    print("-" * 65)

    print(question)

    # --------------------------------------------------------
    # Interview loop
    # --------------------------------------------------------

    question_number = 1

    while True:

        print("\n" + "-" * 65)

        answer = input("You: ")

        # Allow user to exit
        if answer.lower() in ["exit", "quit"]:

            print("\nAI: Thank you for participating.")
            print("Interview ended.")

            break

        # Don't accept empty answers
        if not answer.strip():

            print("\nAI: Please provide an answer.")
            continue

        # ----------------------------------------------------
        # Evaluate answer
        # ----------------------------------------------------

        evaluation = evaluate_answer(
            role,
            question,
            answer
        )

        print("\n")
        print("=" * 65)
        print("ANSWER EVALUATION")
        print("=" * 65)

        print(evaluation)

        # ----------------------------------------------------
        # Generate next question
        # ----------------------------------------------------

        question = generate_next_question(
            role,
            question,
            answer,
            evaluation,
            asked_questions
        )

        asked_questions.append(question)

        question_number += 1

        print("\n")
        print("=" * 65)
        print(f"QUESTION {question_number}")
        print("=" * 65)

        print(question)


# ============================================================
# RUN INTERVIEW
# ============================================================

start_interview()



                  AI INTERVIEWER

AI: Welcome! I'm your AI interviewer.

AI: Great. I'll prepare the interview based on that role.

🔍 Searching PostgreSQL + pgvector...
   ✓ Nomic embedding created (768 dimensions)
   ✓ Retrieved 5 relevant questions


-----------------------------------------------------------------
AI INTERVIEWER
-----------------------------------------------------------------
What design principles do you follow when creating an intuitive and user-friendly interface, and how do you ensure that your designs meet the needs of users with varying abilities?

-----------------------------------------------------------------

🧠 Llama 3.2 is evaluating your answer...

🔍 Searching PostgreSQL + pgvector...
   ✓ Nomic embedding created (768 dimensions)
   ✓ Retrieved 3 relevant questions


ANSWER EVALUATION
SCORE: 2/10

STRENGTHS:
None mentioned.

WEAKNESSES:
- The candidate's answer lacks specific design principles, such as user-centered design, simplicity, and accessibil

KeyboardInterrupt: Interrupted by user

In [10]:
answer = rag_chat(
    "What Python questions should I prepare for a backend developer interview?"
)

print(answer)

I can provide guidance on Python interview preparation for a backend developer position.

To answer your question about what to expect in a Python backend developer interview, here are some areas that are commonly tested:

1. Syntax and Basics: Be prepared to demonstrate proficiency in Python syntax, data structures, and control structures.
2. Object-Oriented Programming (OOP) concepts: Understanding classes, objects, inheritance, polymorphism, and encapsulation is crucial.
3. Data Structures: Familiarize yourself with popular data structures like lists, tuples, dictionaries, sets, and their operations.
4. File Input/Output: Be prepared to handle file input/output operations, such as reading and writing files, and handling exceptions.
5. Database Interaction: If the company uses a specific database management system (DBMS), be prepared to interact with it, including SQL queries and data retrieval.
6. Web Frameworks: Familiarize yourself with popular web frameworks like Django or Flask,

In [11]:
import ollama


# ============================================================
# 1. SEARCH QUESTIONS USING NOMIC + PGVECTOR
# ============================================================

def search_questions(query, limit=5):

    print("\n🔍 Searching PostgreSQL + pgvector...")

    query_embedding = model.encode(query).tolist()

    print(f"   ✓ Nomic embedding created ({len(query_embedding)} dimensions)")

    cur.execute("""
        SELECT
            id,
            question,
            category,
            role,
            experience,
            difficulty,
            ideal_answer,
            keywords,
            1 - (embedding <=> %s::vector) AS similarity
        FROM interview_questions
        WHERE embedding IS NOT NULL
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (
        str(query_embedding),
        str(query_embedding),
        limit
    ))

    results = cur.fetchall()

    print(f"   ✓ Retrieved {len(results)} relevant questions")

    return results


# ============================================================
# 2. BUILD CONTEXT
# ============================================================

def build_context(results):

    return "\n\n".join([
        f"""
Question: {row[1]}
Category: {row[2]}
Role: {row[3]}
Experience: {row[4]}
Difficulty: {row[5]}
Ideal Answer: {row[6]}
Keywords: {row[7]}
Similarity: {row[8]:.3f}
"""
        for row in results
    ])


# ============================================================
# 3. GENERATE INTERVIEW QUESTION
# ============================================================

def generate_question(role, previous_answer=None, evaluation=None, asked_questions=None):

    if asked_questions is None:
        asked_questions = []

    # Search based on role for first question
    if previous_answer is None:

        search_query = role

    # Search based on previous answer for adaptive question
    else:

        search_query = f"""
        Role: {role}

        Candidate previous answer:
        {previous_answer}

        Previous evaluation:
        {evaluation}
        """

    results = search_questions(search_query, limit=5)

    context = build_context(results)

    already_asked = "\n".join(
        f"- {q}" for q in asked_questions
    )

    prompt = f"""
You are a professional AI interviewer.

Candidate role:
{role}

Interview knowledge retrieved from PostgreSQL:
{context}

Questions already asked:
{already_asked}

Previous candidate answer:
{previous_answer}

Previous evaluation:
{evaluation}

Your task:

Generate ONE interview question.

Rules:
- Ask exactly ONE question.
- Make it relevant to the candidate's role.
- Do not repeat previous questions.
- Do not give the answer.
- If the previous answer was weak, ask a useful follow-up or test that weak area.
- If the previous answer was strong, increase the difficulty slightly.
- Keep the question realistic for a job interview.
- Return ONLY the question.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"].strip()


# ============================================================
# 4. EVALUATE INDIVIDUAL ANSWER
# ============================================================

def evaluate_answer(role, question, answer):

    print("\n🧠 Evaluating your answer...")

    results = search_questions(question, limit=3)

    context = build_context(results)

    prompt = f"""
You are an expert technical interviewer.

Candidate role:
{role}

Interview question:
{question}

Candidate answer:
{answer}

Relevant interview knowledge:
{context}

Evaluate the candidate's answer.

Return exactly this format:

SCORE: X/10

STRENGTHS:
- strength 1
- strength 2

WEAKNESSES:
- weakness 1
- weakness 2

FEEDBACK:
Short useful feedback for the candidate.

NEXT_DIFFICULTY:
easy / medium / hard

Be fair and evaluate the actual answer.
Do not invent things the candidate did not say.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"].strip()


# ============================================================
# 5. FINAL INTERVIEW EVALUATION
# ============================================================

def final_evaluation(role, interview_history):

    print("\n")
    print("=" * 65)
    print("GENERATING FINAL INTERVIEW EVALUATION")
    print("=" * 65)

    history_text = ""

    for i, item in enumerate(interview_history, 1):

        history_text += f"""
QUESTION {i}:
{item["question"]}

CANDIDATE ANSWER:
{item["answer"]}

INDIVIDUAL EVALUATION:
{item["evaluation"]}

--------------------------------------------------
"""

    prompt = f"""
You are a senior professional interviewer.

The candidate applied for:

{role}

The candidate completed a 5-question interview.

Here is the complete interview:

{history_text}

Create a final interview evaluation.

Return the evaluation in exactly this structure:

FINAL INTERVIEW EVALUATION
==========================

OVERALL SCORE:
X/10

TECHNICAL KNOWLEDGE:
X/10

PROBLEM SOLVING:
X/10

COMMUNICATION:
X/10

STRENGTHS:
- ...
- ...
- ...

WEAKNESSES:
- ...
- ...
- ...

OVERALL FEEDBACK:
Write a concise but useful overall assessment.

RECOMMENDATION:
Strong Hire / Hire / Consider / Weak Consider / Not Recommended

AREAS TO IMPROVE:
- ...
- ...
- ...

Do not invent information.
Base the evaluation only on the candidate's answers.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"].strip()


# ============================================================
# 6. START 5-QUESTION INTERVIEW
# ============================================================

def start_interview():

    print("\n")
    print("=" * 65)
    print("                    AI INTERVIEWER")
    print("=" * 65)

    print("\nAI: Welcome! I will conduct a 5-question interview.")

    # --------------------------------------------------------
    # Ask role
    # --------------------------------------------------------

    role = input(
        "\nAI: What position are you applying for?\nYou: "
    )

    if role.lower() in ["exit", "quit"]:
        print("\nInterview ended.")
        return

    print(f"\nAI: Great. This interview will focus on: {role}")

    # --------------------------------------------------------
    # Interview variables
    # --------------------------------------------------------

    MAX_QUESTIONS = 5

    asked_questions = []

    interview_history = []

    previous_answer = None

    previous_evaluation = None

    # --------------------------------------------------------
    # Five-question interview
    # --------------------------------------------------------

    for question_number in range(1, MAX_QUESTIONS + 1):

        print("\n")
        print("=" * 65)
        print(f"QUESTION {question_number} OF {MAX_QUESTIONS}")
        print("=" * 65)

        # Generate question
        question = generate_question(
            role=role,
            previous_answer=previous_answer,
            evaluation=previous_evaluation,
            asked_questions=asked_questions
        )

        asked_questions.append(question)

        print("\nAI INTERVIEWER:")
        print(question)

        # ----------------------------------------------------
        # Get candidate answer
        # ----------------------------------------------------

        answer = input("\nYou: ")

        if answer.lower() in ["exit", "quit"]:

            print("\nAI: Interview ended early.")
            return

        while not answer.strip():

            print("AI: Please provide an answer.")

            answer = input("\nYou: ")

        # ----------------------------------------------------
        # Evaluate answer
        # ----------------------------------------------------

        evaluation = evaluate_answer(
            role,
            question,
            answer
        )

        print("\n")
        print("-" * 65)
        print("INDIVIDUAL ANSWER EVALUATION")
        print("-" * 65)

        print(evaluation)

        # ----------------------------------------------------
        # Save interview data
        # ----------------------------------------------------

        interview_history.append({
            "question": question,
            "answer": answer,
            "evaluation": evaluation
        })

        # Prepare for next question
        previous_answer = answer
        previous_evaluation = evaluation

    # ========================================================
    # FINAL EVALUATION
    # ========================================================

    final_result = final_evaluation(
        role,
        interview_history
    )

    print("\n")
    print("=" * 65)
    print("                    FINAL RESULT")
    print("=" * 65)

    print(final_result)

    print("\n")
    print("=" * 65)
    print("                INTERVIEW COMPLETE")
    print("=" * 65)


# ============================================================
# RUN
# ============================================================

start_interview()



                    AI INTERVIEWER

AI: Welcome! I will conduct a 5-question interview.

AI: Great. This interview will focus on: software enggineer


QUESTION 1 OF 5

🔍 Searching PostgreSQL + pgvector...
   ✓ Nomic embedding created (768 dimensions)
   ✓ Retrieved 5 relevant questions

AI INTERVIEWER:
How do you handle conflicting opinions from team members with different technical expertise when working on a complex software project?

🧠 Evaluating your answer...

🔍 Searching PostgreSQL + pgvector...
   ✓ Nomic embedding created (768 dimensions)
   ✓ Retrieved 3 relevant questions


-----------------------------------------------------------------
INDIVIDUAL ANSWER EVALUATION
-----------------------------------------------------------------
SCORE: 4/10

STRENGTHS:

- The candidate acknowledges that conflicting opinions can arise in a team setting, showing some awareness of the issue.

WEAKNESSES:

- The answer lacks specific details on how to handle conflicting opinions. It simply s